# Exploração do Endpoint de Despesas (CEAP)
Testar o endpoint /deputados/{id}/despesas para entender estrutura e parâmetros.

In [0]:
import requests
import json

# Usar um deputado que ja conhecemos
id_deputado = 204379  # Acácio Favacho

url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}/despesas"
params = {
    "ano": 2024,
    "itens": 5,
    "ordem": "ASC",
    "ordenarPor": "dataDocumento"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Total de despesas retornadas: {len(dados['dados'])}")
    print(json.dumps(dados, indent=2, ensure_ascii=False)[:3000])
    
    for link in dados.get('links', []):
        print(f"  {link['rel']}: {link['href']}")
else:
    print(f"Erro: {response.text}")

In [0]:
# Ver quantas paginas de despesas o deputado tem em 2024
import requests

id_deputado = 204379
url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}/despesas"
params = {
    "ano": 2024,
    "itens": 100
}

response = requests.get(url, params=params)
dados = response.json()
print(f"Despesas na primeira pagina: {len(dados['dados'])}")

for link in dados.get('links', []):
    print(f"  {link['rel']}: {link['href']}")

# Coleta de Despesas (Amostra de 50 deputados)

## Decisão técnica
Devido ao volume de dados, optei por uma amostra representativa em vez da coleta completa.
Cada deputado gera ~400 despesas por ano. Para 564 deputados seriam ~225.000 registros,
exigindo mais de 1.000 chamadas à API, o que tornaria o processo inviável no ambiente atual.

## Critérios da amostra
- Diversos partidos e estados
- Dados ordenados por partido e UF para garantir representatividade
- Período: ano de 2024 (mais recente com dados completos)

## Observação
O pipeline está preparado para escalar: basta aumentar o LIMIT na query ou remover
a amostra para coletar todos os deputados quando houver infraestrutura adequada.

In [0]:
import requests
import time

# Selecionar deputados de 2024 (usando tabela prata)
df_amostra = spark.sql("""
    SELECT DISTINCT id_deputado, nome_deputado, sigla_partido, sigla_uf
    FROM prata.eventos_presenca
    WHERE ano = 2024
    ORDER BY sigla_partido, sigla_uf
""")
deputados_amostra = df_amostra.collect()
print(f"Deputados na amostra: {len(deputados_amostra)}")

# Coletar despesas de cada um em 2024
despesas = []
erros = []
inicio = time.time()

for i, dep in enumerate(deputados_amostra, start=1):
    id_dep = dep['id_deputado']
    pagina = 1
    
    while True:
        url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_dep}/despesas"
        params = {
            "ano": 2024,
            "itens": 100,
            "pagina": pagina
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            erros.append(id_dep)
            break
        
        dados = response.json()
        despesas_pagina = dados.get('dados', [])
        
        if not despesas_pagina:
            break
        
        for desp in despesas_pagina:
            despesas.append((
                id_dep,
                dep['nome_deputado'],
                dep['sigla_partido'],
                dep['sigla_uf'],
                desp.get('ano'),
                desp.get('mes'),
                desp.get('tipoDespesa'),
                desp.get('dataDocumento'),
                desp.get('valorDocumento'),
                desp.get('valorLiquido'),
                desp.get('valorGlosa'),
                desp.get('nomeFornecedor'),
                desp.get('cnpjCpfFornecedor'),
                desp.get('urlDocumento')
            ))
        
        links = dados.get('links', [])
        tem_proxima = any(link['rel'] == 'next' for link in links)
        
        if not tem_proxima:
            break
        
        pagina += 1
        time.sleep(0.2)
    
    if i % 10 == 0 or i == 1:
        decorrido = time.time() - inicio
        print(f"  {i}/{len(deputados_amostra)} deputados | {len(despesas)} despesas | {decorrido:.0f}s")

print(f"\nConcluido!")
print(f"Total de despesas: {len(despesas)}")
print(f"Deputados com erro: {len(erros)}")

In [0]:
import json
from datetime import datetime

data_coleta = datetime.now().isoformat()
dados_bronze_desp = []

for d in despesas:
    linha = (
        d[0], d[1], d[2], d[3], d[4], d[5], d[6], d[7], d[8], d[9],
        d[10], d[11], d[12], d[13],
        data_coleta,
        'api_camara_despesas'
    )
    dados_bronze_desp.append(linha)

print(f"Registros preparados: {len(dados_bronze_desp)}")

In [0]:
%sql
DROP TABLE IF EXISTS bronze.despesas

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("id_deputado", LongType(), True),
    StructField("nome_deputado", StringType(), True),
    StructField("sigla_partido", StringType(), True),
    StructField("sigla_uf", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("mes", IntegerType(), True),
    StructField("tipo_despesa", StringType(), True),
    StructField("data_documento", StringType(), True),
    StructField("valor_documento", DoubleType(), True),
    StructField("valor_liquido", DoubleType(), True),
    StructField("valor_glosa", DoubleType(), True),
    StructField("nome_fornecedor", StringType(), True),
    StructField("cnpj_cpf_fornecedor", StringType(), True),
    StructField("url_documento", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df_desp = spark.createDataFrame(dados_bronze_desp, schema=schema)

df_desp.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.despesas")

print(f"Tabela bronze.despesas criada com {df_desp.count()} registros.")

In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT id_deputado) AS deputados,
    COUNT(DISTINCT tipo_despesa) AS categorias,
    ROUND(SUM(valor_liquido), 2) AS total_reembolsado,
    ROUND(AVG(valor_liquido), 2) AS media_despesa
FROM bronze.despesas